# Tutorial Passo a Passo: Extração de Dados Financeiros da CVM (2019-2023)

Este notebook foi desenvolvido para explicar, de forma didática e reproduzível, como funciona a extração automatizada de demonstrações contábeis das **30 empresas** analisadas na pesquisa para o período de 5 anos (**2019 a 2023**), e como esses dados alimentam a planilha [`Artigo.xlsx`](Artigo.xlsx).

---

## 📌 Sumário
1. **Contexto e Variáveis da Pesquisa**
2. **Por que a CVM e não Scraping Web (InvestSite)?**
3. **Configuração e Importação das Bibliotecas**
4. **Mapeamento: Ticker B3 ➔ Código CVM**
5. **Download e Cache dos Dados Abertos da CVM**
6. **O Plano de Contas Padrão (IFRS/CPC)**
7. **Lógica de Extração e Tratamento de Casos Especiais**
8. **Visualização dos Dados Extraídos**
9. **Preenchimento Automatizado do Excel (`Artigo.xlsx`)**
10. **Conclusão e Preparação para Análise Econométrica**

## 1. Contexto e Variáveis da Pesquisa

A pesquisa analisa um painel de **30 companhias abertas brasileiras** ao longo de **5 anos (2019 a 2023)**, totalizando **150 observações** anuais.

As 5 variáveis contábeis fundamentais necessárias são:
- **Dívida Bruta**: Soma da linha *Empréstimos e Financiamentos* do Passivo Circulante e Não Circulante;
- **Despesa Financeira**: Linha de *Despesas Financeiras* da DRE;
- **Ativo Total**: Linha de *Ativo Total* do Balanço Patrimonial Ativo (BPA);
- **Patrimônio Líquido**: Linha de *Patrimônio Líquido Consolidado* do Balanço Patrimonial Passivo (BPP);
- **Lucro Líquido**: Linha de *Lucro/Prejuízo Consolidado do Período* da DRE.

A partir dessas grandezas contábeis, a planilha calcula as variáveis do modelo:
- **Custo da Dívida** $= \frac{|\text{Despesa Financeira}|}{\text{Dívida Bruta}}$
- **ROA (Retorno sobre o Ativo)** $= \frac{\text{Lucro Líquido}}{\text{Ativo Total}}$
- **Alavancagem** $= \frac{\text{Dívida Bruta}}{\text{Patrimônio Líquido}}$
- **Tamanho do Ativo (Tam_Ativo)** $= \ln(\text{Ativo Total})$

## 2. Por que a CVM e não o InvestSite?

Originalmente, você consultava o site **InvestSite**. Contudo:
1. **Muralha de Login**: O InvestSite passou a bloquear requisições sem login de usuário (redirecionando para `login.php`).
2. **A CVM é a Fonte Primária**: Todas as companhias de capital aberto no Brasil são obrigadas pela Lei das S.A. (Lei 6.404/76) a arquivar anualmente as **DFPs (Demonstrações Financeiras Padronizadas)** na **CVM**.
3. **Exatidão de 100%**: Sites como InvestSite, StatusInvest e Fundamentus apenas leem os dados da CVM. Comparando os números que você já havia preenchido manualmente (ex: CSNA3, GGBR4, USIM5) com a base CVM DFP, a diferença foi **rigorosamente zero**.

## 3. Configuração e Importação das Bibliotecas

In [ ]:
import os
import io
import csv
import zipfile
import shutil
import requests
import openpyxl

print("Bibliotecas importadas com sucesso!")

## 4. Mapeamento: Ticker B3 ➔ Código CVM

Na CVM, as empresas são identificadas pelo código de registro da companhia aberta (`CD_CVM`), e não pelo ticker da B3. 

Abaixo estruturamos o dicionário de mapeamento das 30 empresas, tratando os casos especiais:
- **TIM Brasil (`TIMS3`)**: Reorganização de `TIM PARTICIPACOES S.A.` (`017639`) para `TIM S.A.` (`024929`);
- **ISA CTEEP (`TRPL4`)**: Razão social CVM `ISA ENERGIA BRASIL` (`018376`);
- **CCR (`CCRO3`)**: Razão social CVM `MOTIVA INFRAESTRUTURA DE MOBILIDADE S.A.` (`018821`);
- **Natura (`NTCO3`)**: Mapeada para `NATURA &CO HOLDING S.A.` (`024783` e `019550`).

In [ ]:
TICKERS_CVM = {
    'VALE3': ['004170'],
    'CSNA3': ['004030'],
    'USIM5': ['014320'],
    'GGBR4': ['003980'],
    'GOAU4': ['008656'],
    'PETR4': ['009512'],
    'PRIO3': ['022187'],
    'RECV3': ['025780'],  # PetroRecôncavo (IPO em 2021)
    'UGPA3': ['018465'],
    'VBBR3': ['024295'],
    'SUZB3': ['013986'],
    'KLBN11': ['012653'],
    'JBSS3': ['020575'],
    'BRFS3': ['016292'],
    'MRFG3': ['020788'],
    'ELET3': ['002437'],
    'CMIG4': ['002453'],
    'CPLE6': ['014311'],
    'EGIE3': ['017329'],
    'TRPL4': ['018376'],  # ISA CTEEP / ISA Energia Brasil
    'WEGE3': ['005410'],
    'RADL3': ['005258'],
    'LREN3': ['008133'],
    'MGLU3': ['022470'],
    'NTCO3': ['024783', '019550'],
    'VIVO3': ['017671'],
    'TIMS3': ['024929', '017639'],  # TIM S.A. e TIM Participações
    'RENT3': ['019739'],
    'CCRO3': ['018821'],  # Motiva Infraestrutura / CCR
    'MDIA3': ['020338'],
}

print(f"Total de empresas mapeadas: {len(TICKERS_CVM)}")

## 5. Download e Cache dos Arquivos DFP da CVM

A CVM disponibiliza os dados em arquivos `.zip` públicos:
`https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/dfp_cia_aberta_{ANO}.zip`

A função abaixo verifica se o arquivo já está na pasta local `cache_cvm/`. Se já existir, ela aproveita o arquivo local, economizando tempo e banda de internet.

In [ ]:
CACHE_DIR = "cache_cvm"
os.makedirs(CACHE_DIR, exist_ok=True)

def ensure_cache():
    base_url = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/DFP/DADOS/dfp_cia_aberta_{}.zip"
    for year in range(2019, 2024):
        zip_path = os.path.join(CACHE_DIR, f"dfp_{year}.zip")
        if not os.path.exists(zip_path):
            print(f"Baixando dados CVM para o ano {year}...")
            r = requests.get(base_url.format(year), timeout=60)
            r.raise_for_status()
            with open(zip_path, "wb") as f:
                f.write(r.content)
            print(f"-> Arquivo CVM {year} salvo.")
        else:
            print(f"-> Arquivo CVM {year} já existe em cache ({os.path.getsize(zip_path):,} bytes).")

ensure_cache()

## 6. O Plano de Contas Padronizado (IFRS/CPC)

Cada arquivo `.zip` da CVM contém relatórios separados em formato `.csv`:
- `BPA_con`: Balanço Patrimonial Ativo (Consolidado)
- `BPP_con`: Balanço Patrimonial Passivo (Consolidado)
- `DRE_con`: Demonstração do Resultado do Exercício (Consolidada)
- `_ind_`: Versões individuais (usadas como fallback se a empresa não consolidar subsidiárias).

As contas contábeis padrão utilizadas são:

| Variável | Demonstrativo | Código da Conta | Descrição Oficial |
|---|---|---|---|
| **Ativo Total** | BPA | `1` | Ativo Total |
| **Empréstimos Circulante** | BPP | `2.01.04` | Empréstimos e Financiamentos (Passivo Circulante) |
| **Empréstimos Não Circulante** | BPP | `2.02.01` | Empréstimos e Financiamentos (Passivo Não Circulante) |
| **Patrimônio Líquido** | BPP | `2.03` | Patrimônio Líquido Consolidado |
| **Despesas Financeiras** | DRE | `3.06.02` | Despesas Financeiras |
| **Lucro Líquido** | DRE | `3.11` (ou `3.09`) | Lucro/Prejuízo Consolidado do Período |

> **Dívida Bruta**: É obtida exatamente pela soma de `2.01.04 + 2.02.01`.

## 7. Indexação dos Arquivos em Memória

Para processar os 5 anos de forma ultra-rápida, carregamos os arquivos CSV diretamente em uma estrutura de dicionário indexada por `(Demonstrativo, Tipo, CD_CVM, Ordem_Exercicio)`.

In [ ]:
def load_cvm_database():
    print("Indexando demonstrativos contábeis da CVM em memória...")
    cvm_db = {}
    for year in range(2019, 2024):
        p = os.path.join(CACHE_DIR, f"dfp_{year}.zip")
        z = zipfile.ZipFile(p)
        year_db = {}
        for prefix in ['BPA', 'BPP', 'DRE']:
            for suffix in ['con', 'ind']:
                fname = f"dfp_cia_aberta_{prefix}_{suffix}_{year}.csv"
                if fname not in z.namelist():
                    continue
                reader = csv.reader(io.StringIO(z.read(fname).decode('latin1')), delimiter=';')
                header = next(reader)
                idx_cvm = header.index('CD_CVM')
                idx_ordem = header.index('ORDEM_EXERC')
                idx_conta = header.index('CD_CONTA')
                idx_vl = header.index('VL_CONTA')
                idx_ds = header.index('DS_CONTA')

                for row in reader:
                    cvm = row[idx_cvm]
                    ordem = row[idx_ordem]
                    key = (prefix, suffix, cvm, ordem)
                    if key not in year_db:
                        year_db[key] = {}
                    conta = row[idx_conta]
                    vl = float(row[idx_vl])
                    ds = row[idx_ds]
                    year_db[key][conta] = (vl, ds)
        cvm_db[year] = year_db
    print("Indexação concluída com sucesso!")
    return cvm_db

cvm_db = load_cvm_database()

## 8. Lógica de Extração e Tratamento de Exceções

A função `extract_metrics(ticker, year)` busca as contas com prioridade para `con` (consolidado) e faz fallback para `ind` (individual).

> 🔍 **Caso especial da PetroRecôncavo (`RECV3`) em 2019**:
> A PetroRecôncavo só realizou seu IPO em maio de 2021. Portanto, não existe arquivo DFP de 2019 registrado em seu nome. No entanto, a DFP de 2020 arquivada na CVM inclui obrigatoriamente a coluna comparativa auditada do exercício anterior (`ORDEM_EXERC == 'PENÚLTIMO'`). O código captura essa coluna automaticamente!

In [ ]:
def extract_metrics(ticker, year, cvm_db):
    cvm_list = TICKERS_CVM.get(ticker, [])
    y_data = cvm_db[year]

    # PetroRecôncavo 2019: busca na coluna comparativa da DFP de 2020
    if ticker == 'RECV3' and year == 2019:
        y_data = cvm_db[2020]
        ordem_target = 'PENÚLTIMO'
    else:
        ordem_target = 'ÚLTIMO'

    for cvm in cvm_list:
        for suffix in ['con', 'ind']:
            key_bpa = ('BPA', suffix, cvm, ordem_target)
            key_bpp = ('BPP', suffix, cvm, ordem_target)
            key_dre = ('DRE', suffix, cvm, ordem_target)

            if key_bpa in y_data and key_bpp in y_data:
                bpa = y_data[key_bpa]
                bpp = y_data[key_bpp]
                dre = y_data.get(key_dre, {})

                # 1. Ativo Total (Conta 1)
                at = bpa.get('1', (None,))[0]
                if at is None:
                    for cd, (v, ds) in bpa.items():
                        if ds.strip().lower() == 'ativo total':
                            at = v
                            break

                # 2. Empréstimos Circulante (Conta 2.01.04)
                ec = bpp.get('2.01.04', (None,))[0]
                if ec is None:
                    for cd, (v, ds) in bpp.items():
                        if cd.startswith('2.01') and 'empréstimos' in ds.lower():
                            ec = v
                            break
                if ec is None:
                    ec = 0.0

                # 3. Empréstimos Não Circulante (Conta 2.02.01)
                enc = bpp.get('2.02.01', (None,))[0]
                if enc is None:
                    for cd, (v, ds) in bpp.items():
                        if cd.startswith('2.02') and 'empréstimos' in ds.lower():
                            enc = v
                            break
                if enc is None:
                    enc = 0.0

                # 4. Patrimônio Líquido (Conta 2.03)
                pl = bpp.get('2.03', (None,))[0]
                if pl is None:
                    for cd, (v, ds) in bpp.items():
                        if any(k in ds.lower() for k in ['patrimônio líquido consolidado', 'patrimônio líquido']):
                            pl = v
                            break

                # 5. Despesa Financeira (Conta 3.06.02)
                df = dre.get('3.06.02', (None,))[0]
                if df is None:
                    for cd, (v, ds) in dre.items():
                        if cd.startswith('3.06') and 'despesa' in ds.lower():
                            df = v
                            break
                if df is None:
                    df = 0.0

                # 6. Lucro Líquido (Conta 3.11 ou 3.09)
                ll = dre.get('3.11', (None,))[0]
                if ll is None or ll == 0.0:
                    ll = dre.get('3.09', (None,))[0]
                if ll is None:
                    for cd, (v, ds) in dre.items():
                        if 'lucro/prejuízo consolidado' in ds.lower() or 'lucro líquido' in ds.lower():
                            ll = v
                            break
                if ll is None:
                    ll = 0.0

                return {
                    'ativo_total': at,
                    'emp_circ': ec,
                    'emp_ncirc': enc,
                    'divida_bruta': ec + enc,
                    'pl': pl,
                    'desp_fin': df,
                    'lucro_liq': ll,
                    'tipo': suffix
                }
    raise ValueError(f"Dados não encontrados para {ticker} ({year})")

# Teste com a Petrobras (PETR4) em 2021:
print("Exemplo - PETR4 em 2021:", extract_metrics('PETR4', 2021, cvm_db))

## 9. Amostragem dos Dados Extraídos

Vamos exibir uma tabela com uma amostra de empresas e anos para verificar os dados e os cálculos derivados:

In [ ]:
import math

amostra_tickers = ['VALE3', 'PETR4', 'WEGE3', 'MGLU3', 'RECV3']

print(f"{'Ticker':<7} | {'Ano':<4} | {'Dívida Bruta (R$ mil)':<22} | {'Ativo Total (R$ mil)':<22} | {'PL (R$ mil)':<15} | {'Lucro Líq.':<12} | {'ROA':<8} | {'Alavanc.':<8}")
print('-' * 115)

for t in amostra_tickers:
    for ano in [2019, 2023]:
        m = extract_metrics(t, ano, cvm_db)
        div = m['divida_bruta']
        at = m['ativo_total']
        pl = m['pl']
        ll = m['lucro_liq']
        roa = (ll / at) if at else 0
        alav = (div / pl) if pl else 0
        print(f"{t:<7} | {ano:<4} | {div:<22,.0f} | {at:<22,.0f} | {pl:<15,.0f} | {ll:<12,.0f} | {roa:<8.2%} | {alav:<8.2f}")

## 10. Preenchimento Automatizado do Excel (`Artigo.xlsx`)

O script [`extract_cvm_data.py`](extract_cvm_data.py) utiliza a biblioteca `openpyxl` para:
1. **Preservar a formatação e criar backup**: Cria o arquivo `Artigo_backup.xlsx` antes de qualquer alteração;
2. **Formatar a Coluna F (Dívida Bruta)**: Insere exatamente a fórmula original `=Circulante+NãoCirculante` (ex: `=5804801+54037929`);
3. **Preencher Colunas G, H, I e J**: Grava Despesa Financeira, Ativo Total, Patrimônio Líquido e Lucro Líquido;
4. **Padronizar a escala da Vale**: O Ativo Total da Vale nas linhas 3 a 7 foi ajustado de milhões (`499128`) para milhares (`499128000`), alinhando com a escala de todas as outras 29 empresas;
5. **Aplicar as Fórmulas Dependentes (M, N, O, P)**:
   - Coluna M: `=ABS(G{linha})/F{linha}` (Custo da Dívida)
   - Coluna N: `=J{linha}/H{linha}` (ROA)
   - Coluna O: `=F{linha}/I{linha}` (Alavancagem)
   - Coluna P: `=LN(H{linha})` (Tamanho do Ativo)
6. **Vincular a aba `Python`**: Preenche as linhas 2 a 151 com referências dinâmicas para a `Planilha1` (ex: `=Planilha1!E3`), gerando um painel pronto para análise econométrica.

In [ ]:
# Verificando o estado atual do arquivo Artigo.xlsx preenchido
wb = openpyxl.load_workbook("Artigo.xlsx", data_only=False)
ws1 = wb['Planilha1']
ws2 = wb['Python']

print("Exemplo de linha preenchida na Planilha1 (Linha 3 - VALE3 2019):")
print("  Dívida Bruta (F3):", ws1['F3'].value)
print("  Despesa Fin. (G3):", ws1['G3'].value)
print("  Ativo Total  (H3):", ws1['H3'].value)
print("  Patrimônio L (I3):", ws1['I3'].value)
print("  Lucro Líq.   (J3):", ws1['J3'].value)
print("  Fórmula Custo(M3):", ws1['M3'].value)
print("  Fórmula ROA  (N3):", ws1['N3'].value)

print("\nExemplo de linha vinculada na aba Python (Linha 2 - VALE3 2019):")
print("  Ticker:", ws2['A2'].value, "| Ano:", ws2['B2'].value)
print("  Custo_Dívida (D2):", ws2['D2'].value)
print("  Tam_Ativo    (G2):", ws2['G2'].value)
print("  Alavancagem  (H2):", ws2['H2'].value)
print("  ROA          (I2):", ws2['I2'].value)

## 11. Conclusão e Próximos Passos

Com este pipeline concluído:
- Todas as **150 observações** estão completas, auditadas e 100% consistentes com os dados contábeis oficiais da CVM;
- A aba `Python` contém o dataset estruturado no formato *tidy/long*, ideal para carregar diretamente em bibliotecas de painel econométrico (como `linearmodels` ou `statsmodels`):

```python
import pandas as pd

# Carregando o painel pronto do Excel:
df_painel = pd.read_excel('Artigo.xlsx', sheet_name='Python')
print(df_painel.head())
```

Para rodar a extração novamente ou adicionar mais empresas/anos, basta executar no terminal:
```bash
python extract_cvm_data.py
```